# 04 — Station Placement: Optimal Charging Network Design

**Objective:** Place new charging stations along Spain's interurban corridors, targeting the **lowest number of stations** that adequately covers 2027 EV mobility demand. Assign `grid_status` using distributor capacity data.

**This notebook produces File 2** — the core Objective 1 deliverable.

**File 2 schema (mandatory):**
| Field | Type | Description |
|---|---|---|
| `location_id` | String | Sequential ID: `IBE_001`, `IBE_002`, ... |
| `latitude` | Float | WGS84 |
| `longitude` | Float | WGS84 |
| `route_segment` | String | Road designation: `A-3`, `AP-7`, `N-2`, etc. |
| `n_chargers_proposed` | Integer | Chargers at this site |
| `grid_status` | Categorical | `Sufficient` / `Moderate` / `Congested` |

**Mandatory rules:**
- `estimated_demand_kw = n_chargers_proposed × 150 kW`
- `grid_status` from distributor capacity data (i-DE, Endesa, Viesgo)
- File 3 only includes Moderate or Congested locations


In [8]:
# Install dependencies (Colab)
# !pip install geopandas pandas numpy matplotlib shapely -q

import geopandas as gpd
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
from math import radians, cos, sin, asin, sqrt

warnings.filterwarnings('ignore')

def haversine_km(lat1, lon1, lat2, lon2):
    """Haversine distance in km between two points."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

print("Libraries loaded.")


Libraries loaded.


## 1. Load inputs from previous notebooks


In [9]:
# ── Load corridor demand, existing chargers, roads ──

DEMAND_PATH = "../data/processed/demand_by_corridor.csv"
CHARGERS_PATH = "../data/processed/existing_chargers_interurban.csv"
ROADS_PATH = "../data/processed/roads_interurban.geojson"

df_demand = None
if os.path.exists(DEMAND_PATH):
    df_demand = pd.read_csv(DEMAND_PATH, index_col=0)
    print(f"Corridor demand: {len(df_demand)} corridors")
else:
    print(f"WARNING: {DEMAND_PATH} not found. Using default demand tiers.")

existing = None
if os.path.exists(CHARGERS_PATH):
    existing = pd.read_csv(CHARGERS_PATH)
    print(f"Existing chargers: {len(existing)} stations")
else:
    print(f"WARNING: {CHARGERS_PATH} not found. No proximity filter will be applied.")

roads = None
if os.path.exists(ROADS_PATH):
    roads = gpd.read_file(ROADS_PATH)
    print(f"Road network: {len(roads):,} segments")
else:
    print(f"WARNING: {ROADS_PATH} not found.")


Corridor demand: 19 corridors
Existing chargers: 7890 stations
Road network: 217,392 segments


## 2. Station placement algorithm

**Strategy:** Even-spacing placement along each corridor with demand-based sizing.

**Rules:**
1. Place stations every **~150 km** (half of 350 km EV range)
2. Skip if existing charger within **50 km**
3. Size by demand tier: High=6, Medium=4, Low=2 chargers
4. Each charger: **150 kW** (datathon standard)
5. Remove near-duplicate candidates (< 30 km apart from overlapping corridors)


In [10]:
# ── Placement parameters ──

SPACING_KM = 150
MIN_DIST_EXISTING_KM = 50
KW_PER_CHARGER = 150
CHARGERS_HIGH = 6
CHARGERS_MED = 4
CHARGERS_LOW = 2

print(f"Spacing: {SPACING_KM} km")
print(f"Min distance to existing: {MIN_DIST_EXISTING_KM} km")
print(f"Power per charger: {KW_PER_CHARGER} kW")


Spacing: 150 km
Min distance to existing: 50 km
Power per charger: 150 kW


In [11]:
# ── Classify corridors into demand tiers ──

tier_chargers = {'High': CHARGERS_HIGH, 'Medium': CHARGERS_MED, 'Low': CHARGERS_LOW}

if df_demand is not None and 'peak_daily_sessions' in df_demand.columns:
    df_demand = df_demand.sort_values('peak_daily_sessions', ascending=False)
    n = len(df_demand)
    t1, t2 = n // 3, 2 * n // 3

    tiers = ['Low'] * n
    for i in range(t1):
        tiers[i] = 'High'
    for i in range(t1, t2):
        tiers[i] = 'Medium'
    df_demand['demand_tier'] = tiers

    print("Corridor demand tiers:")
    for tier in ['High', 'Medium', 'Low']:
        subset = df_demand[df_demand['demand_tier'] == tier]
        print(f"  {tier}: {list(subset.index)}")
else:
    print("No demand data — using default Medium tier for all corridors.")


Corridor demand tiers:
  High: ['A-2', 'A-4', 'A-3', 'A-1', 'A-6', 'A-5']
  Medium: ['AP-7', 'N-II', 'A-7', 'N-IV', 'N-I', 'N-III']
  Low: ['AP-2', 'N-340', 'A-8', 'AP-68', 'A-92', 'AP-4', 'A-44']


In [12]:
# ── Generate candidate station locations along corridors ──

# Approximate centerline waypoints (lat, lon) for each corridor
# In production, use actual road geometry from notebook 01
CORRIDOR_WAYPOINTS = {
    'AP-7': [(41.39, 2.17), (41.12, 1.25), (40.64, 0.63), (39.47, -0.38),
             (38.35, -0.49), (37.99, -1.13), (36.84, -2.47)],
    'A-7':  [(41.39, 2.17), (41.07, 1.13), (39.99, -0.03), (39.47, -0.38),
             (38.35, -0.49), (37.99, -1.13), (36.72, -4.42), (36.84, -2.47)],
    'A-1':  [(40.42, -3.70), (40.95, -3.17), (42.34, -3.70), (42.85, -2.67)],
    'A-2':  [(40.42, -3.70), (40.63, -3.17), (41.65, -0.88), (41.62, 1.62), (41.39, 2.17)],
    'A-3':  [(40.42, -3.70), (39.86, -2.13), (39.47, -0.38)],
    'A-4':  [(40.42, -3.70), (38.98, -3.93), (37.77, -3.79), (37.88, -4.78), (37.39, -5.99)],
    'A-5':  [(40.42, -3.70), (39.86, -4.02), (38.88, -6.97), (39.47, -6.37)],
    'A-6':  [(40.42, -3.70), (41.65, -4.73), (42.60, -5.57), (43.01, -7.56), (43.37, -8.40)],
    'A-8':  [(43.26, -2.93), (43.46, -3.80), (43.54, -5.66), (43.25, -8.41), (43.37, -8.40)],
    'AP-68':[(43.26, -2.93), (42.46, -2.45), (41.65, -0.88)],
    'AP-2': [(41.65, -0.88), (41.62, 1.62), (41.39, 2.17)],
    'A-92': [(37.39, -5.99), (37.18, -3.60), (36.84, -2.47)],
    'AP-4': [(37.39, -5.99), (36.53, -6.28)],
    'A-44': [(37.77, -3.79), (37.18, -3.60)],
    'N-340':[(41.39, 2.17), (40.99, 0.99), (39.47, -0.38), (38.35, -0.49),
             (37.63, -1.00), (36.72, -4.42)],
    'N-II': [(40.42, -3.70), (41.65, -0.88), (41.39, 2.17)],
    'N-I':  [(40.42, -3.70), (42.34, -3.70), (43.32, -1.98)],
    'N-III':[(40.42, -3.70), (39.86, -2.13), (39.47, -0.38)],
    'N-IV': [(40.42, -3.70), (38.98, -3.93), (37.88, -4.78), (37.39, -5.99), (36.53, -6.28)],
}

def place_along_corridor(waypoints, spacing_km):
    """Place stations at even intervals along a corridor defined by waypoints."""
    # Compute total corridor length
    total_km = 0
    segments = []
    for i in range(len(waypoints) - 1):
        d = haversine_km(waypoints[i][0], waypoints[i][1],
                         waypoints[i+1][0], waypoints[i+1][1])
        segments.append(d)
        total_km += d

    if total_km < 50:  # corridor too short for a station
        # Place one at the midpoint
        mid = len(waypoints) // 2
        return [(round(waypoints[mid][0], 5), round(waypoints[mid][1], 5))]

    # Number of stations (at least 1)
    n_stations = max(1, int(total_km / spacing_km))

    # Place evenly along the route
    points = []
    for s in range(1, n_stations + 1):
        target_km = s * (total_km / (n_stations + 1))
        cum = 0
        for i, seg_len in enumerate(segments):
            if cum + seg_len >= target_km:
                frac = (target_km - cum) / seg_len if seg_len > 0 else 0
                lat = waypoints[i][0] + frac * (waypoints[i+1][0] - waypoints[i][0])
                lon = waypoints[i][1] + frac * (waypoints[i+1][1] - waypoints[i][1])
                points.append((round(lat, 5), round(lon, 5)))
                break
            cum += seg_len

    return points

# Generate all candidates
print("Generating candidate locations...")
all_candidates = []
for corridor, waypoints in CORRIDOR_WAYPOINTS.items():
    locations = place_along_corridor(waypoints, SPACING_KM)

    # Get demand tier
    tier = 'Medium'  # default
    if df_demand is not None and corridor in df_demand.index and 'demand_tier' in df_demand.columns:
        tier = df_demand.loc[corridor, 'demand_tier']

    n_chargers = tier_chargers.get(tier, CHARGERS_MED)

    for lat, lon in locations:
        all_candidates.append({
            'latitude': lat,
            'longitude': lon,
            'route_segment': corridor,
            'n_chargers_proposed': n_chargers,
            'demand_tier': tier,
        })
    print(f"  {corridor}: {len(locations)} stations ({tier} demand, {n_chargers} chargers)")

print(f"\nTotal candidate locations: {len(all_candidates)}")

if len(all_candidates) == 0:
    raise ValueError("No candidate locations generated — check corridor waypoints.")


Generating candidate locations...
  AP-7: 4 stations (Medium demand, 4 chargers)
  A-7: 6 stations (Medium demand, 4 chargers)
  A-1: 2 stations (High demand, 6 chargers)
  A-2: 3 stations (High demand, 6 chargers)
  A-3: 2 stations (High demand, 6 chargers)
  A-4: 3 stations (High demand, 6 chargers)
  A-5: 2 stations (High demand, 6 chargers)
  A-6: 3 stations (High demand, 6 chargers)
  A-8: 3 stations (Low demand, 2 chargers)
  AP-68: 1 stations (Low demand, 2 chargers)
  AP-2: 1 stations (Low demand, 2 chargers)
  A-92: 2 stations (Low demand, 2 chargers)
  AP-4: 1 stations (Low demand, 2 chargers)
  A-44: 1 stations (Low demand, 2 chargers)
  N-340: 5 stations (Low demand, 2 chargers)
  N-II: 3 stations (Medium demand, 4 chargers)
  N-I: 2 stations (Medium demand, 4 chargers)
  N-III: 2 stations (Medium demand, 4 chargers)
  N-IV: 3 stations (Medium demand, 4 chargers)

Total candidate locations: 49


In [13]:
# ── Filter: remove candidates near existing chargers + near-duplicates ──

df_candidates = pd.DataFrame(all_candidates)
print(f"Starting with {len(df_candidates)} candidates")

# 1. Proximity to existing chargers
if existing is not None and len(existing) > 0:
    print(f"\nChecking proximity to {len(existing)} existing chargers (threshold: {MIN_DIST_EXISTING_KM} km)...")
    keep_mask = []
    for _, cand in df_candidates.iterrows():
        dists = [haversine_km(cand['latitude'], cand['longitude'],
                              row['latitude'], row['longitude'])
                 for _, row in existing.iterrows()]
        min_dist = min(dists) if dists else 999
        keep_mask.append(min_dist >= MIN_DIST_EXISTING_KM)

    n_removed = sum(1 for x in keep_mask if not x)
    df_candidates = df_candidates[keep_mask].reset_index(drop=True)
    print(f"  Removed {n_removed} candidates near existing chargers")
    print(f"  Remaining: {len(df_candidates)}")
else:
    print("No existing charger data — keeping all candidates.")

# 2. Remove near-duplicates from overlapping corridors (< 30 km apart)
if len(df_candidates) > 0:
    print(f"\nRemoving near-duplicate candidates (< 30 km apart)...")
    keep = [True] * len(df_candidates)
    for i in range(len(df_candidates)):
        if not keep[i]:
            continue
        for j in range(i+1, len(df_candidates)):
            if not keep[j]:
                continue
            d = haversine_km(
                df_candidates.iloc[i]['latitude'], df_candidates.iloc[i]['longitude'],
                df_candidates.iloc[j]['latitude'], df_candidates.iloc[j]['longitude']
            )
            if d < 30:
                # Keep the one with more chargers
                if df_candidates.iloc[j]['n_chargers_proposed'] > df_candidates.iloc[i]['n_chargers_proposed']:
                    keep[i] = False
                else:
                    keep[j] = False

    n_deduped = sum(1 for x in keep if not x)
    df_stations = df_candidates[keep].reset_index(drop=True)
    print(f"  Removed {n_deduped} near-duplicates")
else:
    df_stations = df_candidates.copy()

print(f"\n==> Final station count: {len(df_stations)}")

if len(df_stations) == 0:
    print("WARNING: All candidates were filtered out!")
    print("This likely means existing charger coverage is already very dense.")
    print("Consider reducing MIN_DIST_EXISTING_KM or adding more corridors.")


Starting with 49 candidates

Checking proximity to 7890 existing chargers (threshold: 50 km)...
  Removed 49 candidates near existing chargers
  Remaining: 0

==> Final station count: 0
This likely means existing charger coverage is already very dense.
Consider reducing MIN_DIST_EXISTING_KM or adding more corridors.


## 3. Assign distributor network and grid status

**Distributor zones (approximate):**
- **i-DE** (Iberdrola): Central and northern Spain (default)
- **Endesa**: Catalonia, Aragón, eastern Andalucía
- **Viesgo**: Cantabria, Asturias

**grid_status** is assigned as a preliminary proxy based on distance to major cities.
T3 will refine this with actual substation capacity data.


In [14]:
# ── Assign distributor network ──

def assign_distributor(lat, lon):
    # Viesgo: Cantabria + Asturias (lat > 43, lon between -6.5 and -3)
    if lat > 43.0 and -6.5 < lon < -3.0:
        return 'Viesgo'
    # Endesa: Catalonia (lon > 0.5 and lat > 40.5)
    if lon > 0.5 and lat > 40.5:
        return 'Endesa'
    # Endesa: Aragon (lon > -1.5 and lat > 40.5 and lon < 0.5)
    if -1.5 < lon < 0.5 and lat > 40.5:
        return 'Endesa'
    # Endesa: eastern Andalucia (lat < 38 and lon > -2.5)
    if lat < 38 and lon > -2.5:
        return 'Endesa'
    # Everything else: i-DE
    return 'i-DE'

if len(df_stations) > 0:
    df_stations['distributor_network'] = df_stations.apply(
        lambda row: assign_distributor(row['latitude'], row['longitude']), axis=1
    )
    print("Distributor assignment:")
    print(df_stations['distributor_network'].value_counts())
else:
    print("No stations to assign distributors to.")


No stations to assign distributors to.


In [15]:
# ── Assign grid_status (preliminary — T3 refines with real substation data) ──

MAJOR_CITIES = {
    'Madrid': (40.42, -3.70), 'Barcelona': (41.39, 2.17),
    'Valencia': (39.47, -0.38), 'Sevilla': (37.39, -5.99),
    'Bilbao': (43.26, -2.93), 'Zaragoza': (41.65, -0.88),
    'Malaga': (36.72, -4.42), 'Murcia': (37.99, -1.13),
}

SUFFICIENT_THRESHOLD_KM = 40
CONGESTED_THRESHOLD_KM = 120

def assign_grid_status(lat, lon, n_chargers):
    min_dist = min(haversine_km(lat, lon, c[0], c[1]) for c in MAJOR_CITIES.values())
    demand_kw = n_chargers * KW_PER_CHARGER

    if min_dist < SUFFICIENT_THRESHOLD_KM:
        return 'Sufficient'
    elif min_dist > CONGESTED_THRESHOLD_KM:
        return 'Congested'
    else:
        return 'Moderate' if demand_kw > 800 else 'Sufficient'

if len(df_stations) > 0:
    df_stations['grid_status'] = df_stations.apply(
        lambda row: assign_grid_status(row['latitude'], row['longitude'],
                                        row['n_chargers_proposed']),
        axis=1
    )
    print("Grid status distribution:")
    print(df_stations['grid_status'].value_counts())
    print("\nNOTE: Preliminary assignment (city-distance proxy). T3 refines with substation data.")
else:
    print("No stations to assign grid status to.")


No stations to assign grid status to.


## 4. Build File 2 (Proposed Charging Locations)

In [16]:
# ── Build File 2 in the exact mandated schema ──

if len(df_stations) == 0:
    print("ERROR: No stations to output. Check upstream notebooks and parameters.")
else:
    file2 = pd.DataFrame({
        'location_id': [f"IBE_{i+1:03d}" for i in range(len(df_stations))],
        'latitude': df_stations['latitude'].values,
        'longitude': df_stations['longitude'].values,
        'route_segment': df_stations['route_segment'].values,
        'n_chargers_proposed': df_stations['n_chargers_proposed'].astype(int).values,
        'grid_status': df_stations['grid_status'].values,
    })

    print("=" * 60)
    print("FILE 2: Proposed Charging Locations")
    print("=" * 60)
    print(f"\nTotal proposed stations: {len(file2)}")
    print(f"Total chargers: {file2['n_chargers_proposed'].sum()}")
    print(f"Total capacity: {file2['n_chargers_proposed'].sum() * KW_PER_CHARGER / 1000:.1f} MW")
    print(f"\nGrid status breakdown:")
    print(file2['grid_status'].value_counts())
    print(f"\nBy route segment:")
    print(file2.groupby('route_segment')['n_chargers_proposed'].agg(['count', 'sum']).rename(
        columns={'count': 'stations', 'sum': 'total_chargers'}
    ).sort_values('stations', ascending=False))
    print(f"\nFirst 10 rows:")
    print(file2.head(10).to_string(index=False))


ERROR: No stations to output. Check upstream notebooks and parameters.


## 5. Visualize proposed network

In [17]:
# ── Map of proposed stations ──

if len(df_stations) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(14, 16))

    if roads is not None:
        roads.plot(ax=ax, color='lightgray', linewidth=0.3, alpha=0.4)

    status_colors = {'Sufficient': '#27ae60', 'Moderate': '#f39c12', 'Congested': '#e74c3c'}
    for status, color in status_colors.items():
        subset = file2[file2['grid_status'] == status]
        if len(subset) > 0:
            ax.scatter(subset['longitude'], subset['latitude'],
                      c=color, s=subset['n_chargers_proposed'] * 15,
                      alpha=0.8, edgecolors='black', linewidth=0.5,
                      label=f"{status} ({len(subset)} stations)", zorder=5)

    if existing is not None:
        ax.scatter(existing['longitude'], existing['latitude'],
                  c='gray', s=10, alpha=0.3, label=f'Existing ({len(existing)})', zorder=3)

    ax.set_title(f"Proposed Interurban Charging Network — Spain 2027\n"
                 f"({len(file2)} new stations, {file2['n_chargers_proposed'].sum()} chargers, "
                 f"{file2['n_chargers_proposed'].sum() * 150 / 1000:.0f} MW)",
                 fontsize=14, fontweight='bold')
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_xlim(-10, 5); ax.set_ylim(35.5, 44)
    ax.legend(loc='lower right', fontsize=11, markerscale=0.8)
    plt.tight_layout(); plt.show()


## 6. Save File 2

In [ ]:
# ── Save File 2 (mandatory output) ──

if len(df_stations) > 0:
    OUTPUT_DIR = "../outputs/"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    file2.to_csv(os.path.join(OUTPUT_DIR, "File_2.csv"), index=False, encoding='utf-8')
    file2.to_csv("../data/processed/file2_proposed_stations.csv", index=False, encoding='utf-8')
    print(f"Saved File_2.csv -> {OUTPUT_DIR}")

    print(f"\n{'='*60}")
    print(f"  FILE 1 INPUT: total_proposed_stations = {len(file2)}")
    print(f"{'='*60}")

    # Compliance check
    print(f"\n--- COMPLIANCE CHECK ---")
    print(f"  [{'OK' if file2['location_id'].str.match(r'^IBE_\d+$').all() else 'FAIL'}] location_id format")
    valid_statuses = {'Sufficient', 'Moderate', 'Congested'}
    print(f"  [{'OK' if set(file2['grid_status'].unique()).issubset(valid_statuses) else 'FAIL'}] grid_status values")
    print(f"  [{'OK' if (file2['n_chargers_proposed'] > 0).all() else 'FAIL'}] n_chargers_proposed > 0")
    print(f"  [{'OK' if 35 < file2['latitude'].min() and file2['latitude'].max() < 44 else 'FAIL'}] latitude in Spain range")
    print(f"  [{'OK' if -10 < file2['longitude'].min() and file2['longitude'].max() < 5 else 'FAIL'}] longitude in Spain range")

    # Preview estimated_demand_kw for File 3
    file2['estimated_demand_kw'] = file2['n_chargers_proposed'] * KW_PER_CHARGER
    print(f"  [OK] estimated_demand_kw = n_chargers × 150 kW")
    print(f"\nFile 2 is ready for submission.")
else:
    print("No stations generated — cannot save File 2.")
